# Programming in Python II - Final Coding Project

Author: Mojtaba Akhundzadah


Student ID: K12443705


Save this file under a file name in the format k+matriculation_number.ipynb, e.g. *k1234567.ipynb*. Remember that for final submission all code cells must run without errors and all cells have to be evaluated.

The code cells are a basic scaffold - you can of course add new code cells if necessary. However, stick to the overall structure of the template to facilitate grading. Ensure to comment your code and structure it reasonably.

In [2]:
import sys

!{sys.executable} -m pip install matplotlib scikit-learn pillow pandas numpy torch torchvision shiny

  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached starlette-1.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached mdit_py_plugins-0.5.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached linkify_it_py-2.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached asgiref-3.11.1-py3-none-any.whl.metadata (9.3 kB)
  Using cached watchfiles-1.1.1-cp313-cp313-win_amd64.whl.metadata (5.0 kB)
  Using cached questionary-2.1.1-py3-none-any.whl.metadata (5.4 kB)
  Using cached uc_micro_py-2.0.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached anyio-4

In [3]:
# ============================================================
# Final Project: Predicting Satellite Images with CNN and PyTorch
# Student ID: K12443705
# ============================================================

from pathlib import Path
import os
import random
import time
import copy
import shutil

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make training as reproducible as possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

DEVICE = device


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"

# The project description calls this folder the test set.
# In this repository/local project it is named public_test_data.
TEST_DIR = PROJECT_DIR / "public_test_data"
if not TEST_DIR.exists():
    TEST_DIR = PROJECT_DIR / "test_data"

ASSETS_DIR = PROJECT_DIR / "assets"
PLOTS_DIR = ASSETS_DIR / "plots"
WEIGHTS_DIR = ASSETS_DIR / "weights"
APP_DIR = PROJECT_DIR / "app"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = WEIGHTS_DIR / "best_model.pth"
APP_MODEL_PATH = APP_DIR / "best_model.pth"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

print("Project directory:", PROJECT_DIR)
print("Training data folder:", DATA_DIR)
print("Training data exists:", DATA_DIR.exists())
print("Test data folder:", TEST_DIR)
print("Test data exists:", TEST_DIR.exists())
print("Device:", DEVICE)


Project directory: c:\Users\USER\Downloads
Training data folder: c:\Users\USER\Downloads\data
Training data exists: False
Test data folder: c:\Users\USER\Downloads\test_data
Test data exists: False
Device: cuda


In [4]:
# Checking successful setup
print(f"Using device: {device}")
print(f"Plots will be saved to: {PLOTS_DIR}")
print(f"Model weights will be saved to: {MODEL_PATH}")


Using device: cuda
Plots will be saved to: c:\Users\USER\Downloads\assets\plots
Model weights will be saved to: c:\Users\USER\Downloads\assets\weights\best_model.pth


In [5]:
# For visualization in Jupyter notebooks
%matplotlib inline

In [6]:
# The notebook should be opened from the final_project folder.
# We intentionally do not use os.chdir("my_project_folder") because this would break local paths.
print(f"Current working directory: {Path.cwd()}")


Current working directory: c:\Users\USER\Downloads


In [7]:
# Creating required folders if necessary.
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("Required folders are ready.")


Required folders are ready.


## Data Handling and Pre-Processing

In [9]:
def preprocess(data_folder: str) -> tuple[pd.DataFrame, dict]:
    """
    Reads the image dataset from class subfolders and returns:
    1. A DataFrame with columns: folder, file_name, label
    2. A dictionary mapping class names to numeric labels

    The folder structure must be:
    data/
        AnnualCrop/
        Forest/
        ...
    """

    data_path = Path(data_folder)

    if not data_path.exists():
        raise FileNotFoundError(
            f"Data folder not found: {data_path}. "
            "Make sure the local data/ folder exists next to this notebook."
        )

    class_names = sorted([
        folder.name for folder in data_path.iterdir()
        if folder.is_dir()
    ])

    if len(class_names) == 0:
        raise ValueError(f"No class folders found in: {data_path}")

    class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}

    rows = []
    valid_extensions = {".jpg", ".jpeg", ".png"}

    for class_name in class_names:
        class_folder = data_path / class_name
        label = class_to_label[class_name]

        for image_path in sorted(class_folder.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in valid_extensions:
                rows.append({
                    "folder": str(class_folder),
                    "file_name": image_path.name,
                    "label": label
                })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No images found. Please check your local data folder.")

    return df, class_to_label


# Run preprocessing on the local data folder.
dataset, class_to_label = preprocess(DATA_DIR)
label_to_class = {label: class_name for class_name, label in class_to_label.items()}

# Add a readable class-name column for analysis.
dataset["class_name"] = dataset["label"].map(label_to_class)

print("Number of images:", len(dataset))
print("Number of classes:", len(class_to_label))
print("Class to label mapping:")
print(class_to_label)

dataset.head()


FileNotFoundError: Data folder not found: c:\Users\USER\Downloads\data. Make sure the local data/ folder exists next to this notebook.

In [ ]:
# Checking successful dataframe loading.
print(f"Length of dataset: {len(dataset)}")  # Should be 10000 according to the project description.

print("\nDataFrame columns:")
print(dataset.columns.tolist())

print("\nClass distribution:")
print(dataset["class_name"].value_counts().sort_index())

print("\nFirst rows:")
dataset.head()


In [ ]:
# Stratified train/validation split.
# Stratified means that every class has approximately the same proportion in train and validation data.

train_df, val_df = train_test_split(
    dataset,
    test_size=0.20,
    random_state=SEED,
    stratify=dataset["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts().sort_index())


## Exploratory Data Analysis (EDA)

In [ ]:
def load_image_from_row(row: pd.Series) -> Image.Image:
    """Loads an RGB image from one row of the DataFrame."""
    image_path = Path(row["folder"]) / row["file_name"]
    return Image.open(image_path).convert("RGB")


def show_samples(df: pd.DataFrame, num_samples: int = 5) -> None:
    """
    Randomly selects images from the DataFrame, displays them in a grid,
    and saves the plot to assets/plots/random_samples.png.
    """

    sample_df = df.sample(n=num_samples, random_state=SEED).reset_index(drop=True)

    fig, axes = plt.subplots(1, num_samples, figsize=(3 * num_samples, 3))

    if num_samples == 1:
        axes = [axes]

    for idx, ax in enumerate(axes):
        row = sample_df.iloc[idx]
        image = load_image_from_row(row)
        label_name = label_to_class[int(row["label"])]

        ax.imshow(image)
        ax.set_title(f"Label: {label_name}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()

    save_path = PLOTS_DIR / "random_samples.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


show_samples(train_df, num_samples=5)


In [ ]:
def average_pixel_plot(df: pd.DataFrame) -> None:
    """
    Creates a plot showing the distribution of average pixel values
    in the three color channels in the training set.

    The plot is saved to assets/plots/average_pixel_distribution.png.
    """

    red_values = []
    green_values = []
    blue_values = []

    for _, row in df.iterrows():
        image = load_image_from_row(row)
        arr = np.array(image)

        red_values.append(arr[:, :, 0].mean())
        green_values.append(arr[:, :, 1].mean())
        blue_values.append(arr[:, :, 2].mean())

    plt.figure(figsize=(10, 6))

    plt.hist(red_values, bins=40, alpha=0.5, label="Red")
    plt.hist(green_values, bins=40, alpha=0.5, label="Green")
    plt.hist(blue_values, bins=40, alpha=0.5, label="Blue")

    plt.title("Distribution of Average Pixel Values")
    plt.xlabel("Average Pixel Value")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)

    save_path = PLOTS_DIR / "average_pixel_distribution.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


average_pixel_plot(train_df)


In [ ]:
def average_brightness_per_class(df: pd.DataFrame) -> None:
    """
    Creates a boxplot showing the distribution of average brightness
    for each class in the training set.

    Brightness is calculated as the average pixel value over all RGB channels.
    The plot is saved to assets/plots/average_brightness.png.
    """

    brightness_rows = []

    for _, row in df.iterrows():
        image = load_image_from_row(row)
        arr = np.array(image)

        brightness_rows.append({
            "class_name": label_to_class[int(row["label"])],
            "brightness": arr.mean()
        })

    brightness_df = pd.DataFrame(brightness_rows)
    class_names = sorted(brightness_df["class_name"].unique())

    data_to_plot = [
        brightness_df.loc[brightness_df["class_name"] == class_name, "brightness"].values
        for class_name in class_names
    ]

    plt.figure(figsize=(12, 6))
    plt.boxplot(data_to_plot, labels=class_names, showfliers=True)

    plt.title("Average Brightness Distribution per Class")
    plt.xlabel("Class")
    plt.ylabel("Average Brightness")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)

    save_path = PLOTS_DIR / "average_brightness.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


average_brightness_per_class(train_df)


## CNN Implementation and Training

### Dataset class

In [ ]:
class SatelliteDataset(Dataset):
    """
    Custom PyTorch Dataset for satellite images.
    It uses the DataFrame from the preprocessing step.
    """

    def __init__(self, df: pd.DataFrame, transform=None, has_labels: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["folder"]) / row["file_name"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.has_labels:
            label = int(row["label"])
            return image, label

        return image, row["file_name"]


### Data Loaders

In [ ]:
# Transformations for training and validation/test.
# Training uses light augmentation. Validation/test uses deterministic preprocessing only.

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# DataLoaders
BATCH_SIZE = 64
NUM_WORKERS = 0  # safer for Windows and Jupyter notebooks

train_dataset = SatelliteDataset(train_df, transform=train_transform, has_labels=True)
val_dataset = SatelliteDataset(val_df, transform=val_test_transform, has_labels=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


### Model Architecture

In [ ]:
class SatelliteCNN(nn.Module):
    """
    Custom CNN model for 64x64 RGB satellite images.
    No pretrained model and no transfer learning are used.
    """

    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.features = nn.Sequential(
            # Input: 3 x 64 x 64
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.05),
            # 32 x 32 x 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.10),
            # 64 x 16 x 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.15),
            # 128 x 8 x 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.20),
            # 256 x 4 x 4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


num_classes = len(class_to_label)
model = SatelliteCNN(num_classes=num_classes).to(device)

print(model)


### Training Loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Trains the model for one epoch."""

    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


def validate_one_epoch(model, dataloader, criterion, device):
    """Evaluates the model on the validation set for one epoch."""

    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, dim=1)

            running_loss += loss.item() * images.size(0)
            running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_model_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

start_time = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, MODEL_PATH)
        epochs_without_improvement = 0
        print(f"  -> New best model saved with validation accuracy: {best_val_acc:.4f}")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

end_time = time.time()

print(f"\nTraining finished in {(end_time - start_time) / 60:.2f} minutes.")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Best model saved to: {MODEL_PATH}")

# Load the best model weights again.
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
_ = model.eval()

# Also copy the model into the app folder so the Shiny app directory is self-contained.
if MODEL_PATH.exists():
    shutil.copy2(MODEL_PATH, APP_MODEL_PATH)
    print(f"Copied best model to app folder: {APP_MODEL_PATH}")


## Model Evaluation

In [ ]:
def plot_training_curves(history: dict) -> None:
    """
    Plots training/validation loss and validation accuracy side by side.
    The plot is saved to assets/plots/training_curves.png.
    """

    epochs_range = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, history["train_loss"], label="Training Loss")
    axes[0].plot(epochs_range, history["val_loss"], label="Validation Loss")
    axes[0].set_title("Training and Validation Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history["val_acc"], label="Validation Accuracy")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    save_path = PLOTS_DIR / "training_curves.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


plot_training_curves(history)


In [ ]:
def get_predictions(model, dataloader, device):
    """Returns predicted labels and true labels for a dataloader."""

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)


def plot_confusion_matrix(model, dataloader, device) -> None:
    """
    Creates and saves a confusion matrix for the validation set.
    """

    val_preds, val_labels = get_predictions(model, dataloader, device)
    cm = confusion_matrix(val_labels, val_preds)

    class_names = [label_to_class[i] for i in range(len(label_to_class))]

    fig, ax = plt.subplots(figsize=(10, 10))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        ax=ax,
        xticks_rotation=45,
        cmap="Blues",
        values_format="d"
    )

    plt.title("Confusion Matrix on Validation Set")
    plt.tight_layout()

    save_path = PLOTS_DIR / "confusion_matrix.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


plot_confusion_matrix(model, val_loader, device)


In [ ]:
def plot_misclassified_samples(model, dataset, device, num_samples: int = 5) -> None:
    """
    Shows misclassified validation images with true and predicted labels.
    The plot is saved to assets/plots/misclassified_samples.png.
    """

    model.eval()
    misclassified = []

    with torch.no_grad():
        for idx in range(len(dataset)):
            image_tensor, true_label = dataset[idx]

            input_tensor = image_tensor.unsqueeze(0).to(device)
            output = model(input_tensor)
            _, pred_label = torch.max(output, dim=1)

            pred_label = int(pred_label.item())
            true_label = int(true_label)

            if pred_label != true_label:
                misclassified.append((idx, true_label, pred_label))

            if len(misclassified) >= num_samples:
                break

    if len(misclassified) == 0:
        print("No misclassified samples found.")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(4 * len(misclassified), 4))

    if len(misclassified) == 1:
        axes = [axes]

    for ax, (idx, true_label, pred_label) in zip(axes, misclassified):
        row = dataset.df.iloc[idx]
        image = load_image_from_row(row)

        true_name = label_to_class[true_label]
        pred_name = label_to_class[pred_label]

        ax.imshow(image)
        ax.set_title(
            f"True: {true_name}\nPred: {pred_name}",
            fontsize=9
        )
        ax.axis("off")

    plt.tight_layout()

    save_path = PLOTS_DIR / "misclassified_samples.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved plot to: {save_path}")


plot_misclassified_samples(model, val_dataset, device, num_samples=5)


### Test Set

In [ ]:
def natural_image_sort_key(path: Path):
    """Sorts numeric file names like 1.jpg, 2.jpg, 10.jpg in numeric order."""
    try:
        return int(path.stem)
    except ValueError:
        return path.name


def preprocess_test(data_folder: str) -> pd.DataFrame:
    """
    Reads test images from a folder without class subfolders.
    Returns a DataFrame with columns: folder, file_name.
    """

    test_path = Path(data_folder)

    if not test_path.exists():
        raise FileNotFoundError(
            f"Test folder not found: {test_path}. "
            "Make sure public_test_data/ exists locally next to this notebook."
        )

    valid_extensions = {".jpg", ".jpeg", ".png"}
    rows = []

    image_paths = [
        p for p in test_path.iterdir()
        if p.is_file() and p.suffix.lower() in valid_extensions
    ]

    for image_path in sorted(image_paths, key=natural_image_sort_key):
        rows.append({
            "folder": str(test_path),
            "file_name": image_path.name
        })

    test_df = pd.DataFrame(rows)

    if test_df.empty:
        raise ValueError("No test images found.")

    return test_df


test_df = preprocess_test(TEST_DIR)

print("Number of public test images:", len(test_df))
test_df.head()


In [ ]:
test_dataset = SatelliteDataset(
    test_df,
    transform=val_test_transform,
    has_labels=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Test batches:", len(test_loader))


In [ ]:
def predict_test_data(model, dataloader, device, label_to_class: dict) -> pd.DataFrame:
    """
    Predicts classes for the public test data and returns a submission DataFrame.
    """

    model.eval()
    submission_rows = []

    with torch.no_grad():
        for images, file_names in dataloader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)

            preds = preds.cpu().numpy()

            for file_name, pred_label in zip(file_names, preds):
                class_name = label_to_class[int(pred_label)]
                submission_rows.append({
                    "file_name": file_name,
                    "predicted_class": class_name
                })

    return pd.DataFrame(submission_rows)


submission_df = predict_test_data(
    model,
    test_loader,
    device,
    label_to_class
)

submission_df.head()


In [ ]:
# Create submission.csv for the challenge server.
# Required format: file_name,predicted_class_name
# Important: no header and class names as text.

submission_df.to_csv(
    SUBMISSION_PATH,
    index=False,
    header=False
)

print(f"Submission file saved to: {SUBMISSION_PATH}")
print("Preview:")
print(submission_df.head(10).to_string(index=False))


## Shiny Web Application


In [ ]:
app_code = '# Mojtaba Akhundzadah - Student ID: K12443705\n\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torchvision import transforms\n\nfrom shiny.express import input, render, ui\n\n\n# ------------------------------------------------------------\n# Paths and device\n# ------------------------------------------------------------\n\nAPP_DIR = Path(__file__).parent\nPROJECT_DIR = APP_DIR.parent\n\nMODEL_PATH = APP_DIR / "best_model.pth"\nif not MODEL_PATH.exists():\n    MODEL_PATH = PROJECT_DIR / "assets" / "weights" / "best_model.pth"\n\nif torch.cuda.is_available():\n    DEVICE = torch.device("cuda")\nelif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n    DEVICE = torch.device("mps")\nelse:\n    DEVICE = torch.device("cpu")\n\n\n# ------------------------------------------------------------\n# Class names\n# This order must match the sorted class-folder order in the notebook.\n# ------------------------------------------------------------\n\nCLASS_NAMES = [\n    "AnnualCrop",\n    "Forest",\n    "HerbaceousVegetation",\n    "Highway",\n    "Industrial",\n    "Pasture",\n    "PermanentCrop",\n    "Residential",\n    "River",\n    "SeaLake",\n]\n\n\n# ------------------------------------------------------------\n# Model architecture\n# This must match the notebook model exactly.\n# ------------------------------------------------------------\n\nclass SatelliteCNN(nn.Module):\n    def __init__(self, num_classes: int = 10):\n        super().__init__()\n\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 32, kernel_size=3, padding=1),\n            nn.BatchNorm2d(32),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(32, 32, kernel_size=3, padding=1),\n            nn.BatchNorm2d(32),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Dropout2d(0.05),\n\n            nn.Conv2d(32, 64, kernel_size=3, padding=1),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(64, 64, kernel_size=3, padding=1),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Dropout2d(0.10),\n\n            nn.Conv2d(64, 128, kernel_size=3, padding=1),\n            nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(128, 128, kernel_size=3, padding=1),\n            nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Dropout2d(0.15),\n\n            nn.Conv2d(128, 256, kernel_size=3, padding=1),\n            nn.BatchNorm2d(256),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(256, 256, kernel_size=3, padding=1),\n            nn.BatchNorm2d(256),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Dropout2d(0.20),\n        )\n\n        self.classifier = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(256 * 4 * 4, 512),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.50),\n            nn.Linear(512, 128),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.30),\n            nn.Linear(128, num_classes)\n        )\n\n    def forward(self, x):\n        x = self.features(x)\n        x = self.classifier(x)\n        return x\n\n\n# ------------------------------------------------------------\n# Prediction transform\n# Same as validation/test transform in the notebook.\n# ------------------------------------------------------------\n\nprediction_transform = transforms.Compose([\n    transforms.ToTensor(),\n    transforms.Normalize(\n        mean=[0.485, 0.456, 0.406],\n        std=[0.229, 0.224, 0.225]\n    )\n])\n\n\n# ------------------------------------------------------------\n# Load model\n# ------------------------------------------------------------\n\nmodel = SatelliteCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)\n\nstate_dict = torch.load(MODEL_PATH, map_location=DEVICE)\n_ = model.load_state_dict(state_dict)\n_ = model.eval()\n\n\n# ------------------------------------------------------------\n# User interface\n# ------------------------------------------------------------\n\nui.page_opts(title="Satellite Image Classifier", fillable=True)\n\nui.h2("Satellite Image Classifier")\nui.p("Upload a 64x64 RGB satellite image. The CNN model predicts the terrain class.")\n\nui.input_file(\n    "image_file",\n    "Select satellite image",\n    accept=[".jpg", ".jpeg", ".png"],\n    multiple=False\n)\n\nui.hr()\n\n\n@render.image\ndef uploaded_image():\n    file_info = input.image_file()\n\n    if not file_info:\n        return None\n\n    image_path = file_info[0]["datapath"]\n\n    return {\n        "src": image_path,\n        "width": "300px",\n        "height": "300px",\n        "alt": "Uploaded satellite image"\n    }\n\n\ndef predict_uploaded_image():\n    file_info = input.image_file()\n\n    if not file_info:\n        return None\n\n    image_path = file_info[0]["datapath"]\n\n    image = Image.open(image_path).convert("RGB")\n    image_tensor = prediction_transform(image).unsqueeze(0).to(DEVICE)\n\n    with torch.no_grad():\n        logits = model(image_tensor)\n        probabilities = F.softmax(logits, dim=1).squeeze(0)\n\n    probabilities_cpu = probabilities.cpu().numpy()\n    predicted_index = int(probabilities.argmax().item())\n    predicted_class = CLASS_NAMES[predicted_index]\n    confidence = float(probabilities[predicted_index].item())\n\n    probability_df = pd.DataFrame({\n        "Class": CLASS_NAMES,\n        "Probability": probabilities_cpu\n    })\n\n    probability_df["Probability"] = probability_df["Probability"].round(4)\n    probability_df = probability_df.sort_values(\n        by="Probability",\n        ascending=False\n    ).reset_index(drop=True)\n\n    return predicted_class, confidence, probability_df\n\n\n@render.ui\ndef prediction_card():\n    result = predict_uploaded_image()\n\n    if result is None:\n        return ui.div("Please upload an image first.")\n\n    predicted_class, confidence, _ = result\n\n    return ui.div(\n        ui.h4("Prediction"),\n        ui.p(f"Predicted class: {predicted_class}"),\n        ui.p(f"Confidence: {confidence * 100:.2f}%")\n    )\n\n\n@render.data_frame\ndef probability_table():\n    result = predict_uploaded_image()\n\n    if result is None:\n        return pd.DataFrame({\n            "Class": CLASS_NAMES,\n            "Probability": [0.0] * len(CLASS_NAMES)\n        })\n\n    _, _, probability_df = result\n    return probability_df\n'

app_path = APP_DIR / "app.py"
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"Shiny app written to: {app_path}")
print("Run locally with:")
print("shiny run --reload --launch-browser app/app.py")
